# 03 — WARC HTML Extraction

For each URL in our training set (WDC records), fetch the raw HTML from
Common Crawl WARC files using HTTP range requests.

**Why**: WDC gives us (URL, JSON-LD) but not the HTML. We need HTML to create
the input side of our training pairs (screenshot + HTML → JSON-LD).

**Tip**: Run this from EC2 us-east-1 for free S3 egress. Running locally works
but incurs CC bandwidth costs (~$0.09/GB).

**Output**: `data/raw/html/` — one .html file per URL

In [1]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('../.env', override=True)

HTML_DIR = Path('../data/raw/html')
HTML_DIR.mkdir(parents=True, exist_ok=True)

print('Output directory:', HTML_DIR)

Output directory: ../data/raw/html


## Load WDC Records with WARC Pointers

First, we need to join our WDC records (URL + JSON-LD) with the CC index
to get WARC file pointers (filename, offset, length).

In [2]:
from src.common_crawl import query_cc_index

# Load WDC records
wdc_records = []
for fname in ['../data/raw/wdc_ie_records.jsonl', '../data/raw/wdc_global_records.jsonl']:
    if Path(fname).exists():
        with open(fname) as f:
            wdc_records.extend([json.loads(line) for line in f])

print(f'Loaded {len(wdc_records)} WDC records')

# Extract unique URLs we need to fetch
urls_needed = list({rec['source_url'] for rec in wdc_records if rec.get('source_url')})
print(f'Unique URLs to fetch: {len(urls_needed)}')

Loaded 0 WDC records
Unique URLs to fetch: 0


In [3]:
# For each URL, query CC index to get WARC pointers
# Batch query: query each URL individually (or use domain-level query + filter)
# For large batches, the Athena/DuckDB approach from notebook 01 is more efficient

CRAWL = 'CC-MAIN-2026-08'

# Quick approach: query individual URLs
# For scale: use DuckDB/Athena with a list of URLs in a WHERE IN clause

def get_warc_pointer(url: str, crawl: str = CRAWL) -> dict | None:
    """Get WARC file pointer for a single URL from CC index."""
    records = query_cc_index(url, crawl=crawl, match_type='exact', limit=1)
    return records[0] if records else None

# Test with one URL
if urls_needed:
    test_pointer = get_warc_pointer(urls_needed[0])
    print('Test WARC pointer:', test_pointer)

## Batch WARC Fetch

Fetch HTML for each URL using HTTP range requests against CC S3.

In [ ]:
from src.common_crawl import batch_fetch_warc
from tqdm.auto import tqdm
import time

fetch_records = []
failed_lookups = []

sample_urls = urls_needed[:100]

print(f'Looking up WARC pointers for {len(sample_urls)} URLs...')
for url in tqdm(sample_urls, desc="CC index lookup"):
    pointer = get_warc_pointer(url)
    if pointer:
        fetch_records.append(pointer)
    else:
        failed_lookups.append(url)
    time.sleep(0.2)

print(f'Found WARC pointers: {len(fetch_records)}')
print(f'Not in index: {len(failed_lookups)}')

In [5]:
# Fetch HTML from WARC files
fetched = list(batch_fetch_warc(
    records=fetch_records,
    output_dir=str(HTML_DIR),
    max_records=100,
))

print(f'\nFetched HTML for {len(fetched)} pages')
if fetched:
    sample = fetched[0]
    print(f'Sample URL: {sample["url"]}')
    print(f'HTML length: {len(sample["html"])} chars')
    print('HTML preview:', sample['html'][:200])

Fetching WARC records: 0it [00:00, ?it/s]


Fetched HTML for 0 pages


## Quality Check: HTML + Schema Pairing

In [ ]:
from src.schema_validator import extract_jsonld_from_html, validate_jsonld

# Sanity check: does the fetched HTML still contain the JSON-LD we expect?
# (Sometimes the CC WARC record is older than the WDC extraction — minor inconsistency)

consistency_results = []
for item in fetched[:20]:
    url = item['url']
    html = item['html']
    
    extracted = extract_jsonld_from_html(html)
    consistency_results.append({
        'url': url,
        'has_jsonld_in_html': len(extracted) > 0,
        'n_jsonld_blocks': len(extracted),
    })

import pandas as pd
df = pd.DataFrame(consistency_results)
print(df['has_jsonld_in_html'].value_counts())
print(f'\n{df["has_jsonld_in_html"].mean()*100:.1f}% of fetched pages still contain JSON-LD')

In [ ]:
# Save paired (url, html_path, wdc_jsonld) manifest for next step
# Map fetched HTML back to WDC records

wdc_by_url = {rec['source_url']: rec for rec in wdc_records}

manifest = []
for item in fetched:
    url = item['url']
    if url in wdc_by_url:
        manifest.append({
            'url': url,
            'html_path': str(HTML_DIR / f"{url.replace('://', '_').replace('/', '_')[:100]}.html"),
            'wdc_jsonld': wdc_by_url[url].get('jsonld'),
            'schema_type': wdc_by_url[url].get('schema_type'),
            'property_count': wdc_by_url[url].get('property_count'),
        })

manifest_path = Path('../data/raw/warc_manifest.jsonl')
with open(manifest_path, 'w') as f:
    for entry in manifest:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f'Saved manifest: {len(manifest)} paired examples to {manifest_path}')
print('\nNext step: 04_screenshot_rendering.ipynb')